In [ ]:
import os
import calendar
import time
import random
import zipfile
from pathlib import Path
from copy import deepcopy

import cdsapi
from tqdm.auto import tqdm


c:\Users\duruenaramirez\AppData\Local\miniforge3\envs\PHLFlood\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Point cdsapi to the .cdsapirc that lives INSIDE your repo
# (cdsapi otherwise expects it in $HOME/.cdsapirc) 
rc_path = Path(".") / ".cdsapirc"
assert rc_path.exists(), f"Repo .cdsapirc not found: {rc_path.resolve()}"

os.environ["CDSAPI_RC"] = str(rc_path.resolve())

# Optional safety: ensure you’re using EWDS endpoint (matches EWDS docs)
os.environ["CDSAPI_URL"] = "https://ewds.climate.copernicus.eu/api"  

client = cdsapi.Client()

In [3]:
dataset = "cems-glofas-historical"

SYSTEM_VERSION = "version_4_0"
PRODUCT_TYPE = "consolidated"
VARIABLE = "river_discharge_in_the_last_24_hours"
HYDRO_MODEL = "lisflood"

EXTRACT = True  # set True if you want extraction

# whole-country bbox in [N, W, S, E] 
AREA = [35, 63, 4, 131]

START_YEAR = 1979
END_YEAR   = 2025  # inclusive

months = [f"{m:02d}" for m in range(1, 13)]
days   = [f"{d:02d}" for d in range(1, 32)]

OUT_DIR = (
    Path("data/raw/glofas/historical")
    / SYSTEM_VERSION
    / PRODUCT_TYPE
    / "discharge"
    / "grib2"
    / f"area_{AREA[0]}_{AREA[1]}_{AREA[2]}_{AREA[3]}"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def target_zip_for_year(year: int) -> Path:
    # Match your naming convention + add .zip extension
    # Example you gave: glofas_historical_version_4_0_consolidated_river_discharge_in_the_last_24_hours_2025
    # We'll store as ..._2025.zip (more explicit / safer).
    stem = f"glofas_historical_{SYSTEM_VERSION}_{PRODUCT_TYPE}_{VARIABLE}_{year}"
    return OUT_DIR / f"{stem}.zip"

print("Output folder:", OUT_DIR)
print("Example target:", target_zip_for_year(2025))


Output folder: data\raw\glofas\historical\version_4_0\consolidated\discharge\grib2\area_35_63_4_131
Example target: data\raw\glofas\historical\version_4_0\consolidated\discharge\grib2\area_35_63_4_131\glofas_historical_version_4_0_consolidated_river_discharge_in_the_last_24_hours_2025.zip


In [4]:
BASE_REQUEST = {
    "system_version": [SYSTEM_VERSION],
    "hydrological_model": [HYDRO_MODEL],
    "product_type": [PRODUCT_TYPE],
    "variable": [VARIABLE],
    "hmonth": months,
    "hday": days,
    "data_format": "grib2",
    "download_format": "zip",
    "area": AREA,
}

def is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    # quick structural check
    if not zipfile.is_zipfile(path):
        return False
    # deeper check (optional but good): try listing contents
    try:
        with zipfile.ZipFile(path, "r") as zf:
            _ = zf.namelist()[:5]
        return True
    except Exception:
        return False

def retrieve_with_retries(dataset: str, request: dict, target: Path, max_attempts: int = 5):
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            # 3rd argument writes directly to your target path :contentReference[oaicite:6]{index=6}
            client.retrieve(dataset, request, str(target))
            return
        except Exception as e:
            last_err = e
            # Backoff with jitter
            sleep_s = min(120, 5 * attempt) + random.uniform(0, 2)
            print(f"[Attempt {attempt}/{max_attempts}] Error: {e}\nSleeping {sleep_s:.1f}s then retrying...")
            time.sleep(sleep_s)
    raise last_err



In [5]:
years = list(range(START_YEAR, END_YEAR + 1))

for year in tqdm(years, desc="Downloading yearly ZIPs"):
    target = target_zip_for_year(year)

    # Support BOTH possibilities:
    # - our preferred target "..._YYYY.zip"
    # - your existing files without ".zip" suffix
    legacy_target = Path(str(target)[:-4])  # removes ".zip"
    have_file = is_valid_zip(target) or is_valid_zip(legacy_target)

    if have_file:
        tqdm.write(f"✓ {year} already present (skipping)")
        continue

    # If a previous partial download exists, remove it
    for p in [target, legacy_target]:
        if p.exists() and not is_valid_zip(p):
            tqdm.write(f"⚠ Removing corrupt/partial file: {p.name}")
            try:
                p.unlink()
            except Exception:
                pass

    req = deepcopy(BASE_REQUEST)
    req["hyear"] = [str(year)]

    tqdm.write(f"↓ Downloading {year} → {target.name}")
    retrieve_with_retries(dataset, req, target, max_attempts=5)

    if not is_valid_zip(target):
        raise RuntimeError(f"Downloaded file is not a valid ZIP: {target}")

    tqdm.write(f"✓ Done {year} (size={target.stat().st_size/1e6:.1f} MB)")
    if EXTRACT:
        extract_dir = OUT_DIR / str(year)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target, "r") as zf:
            zf.extractall(extract_dir)
        tqdm.write(f"↳ Extracted to {extract_dir}")


✓ 1979 already present (skipping)
✓ 1980 already present (skipping)
✓ 1981 already present (skipping)
✓ 1982 already present (skipping)
✓ 1983 already present (skipping)


✓ 1984 already present (skipping)
✓ 1985 already present (skipping)
✓ 1986 already present (skipping)
✓ 1987 already present (skipping)


✓ 1988 already present (skipping)
✓ 1989 already present (skipping)
✓ 1990 already present (skipping)
✓ 1991 already present (skipping)


✓ 1992 already present (skipping)
✓ 1993 already present (skipping)
✓ 1994 already present (skipping)
✓ 1995 already present (skipping)
✓ 1996 already present (skipping)


✓ 1997 already present (skipping)
✓ 1998 already present (skipping)
✓ 1999 already present (skipping)
✓ 2000 already present (skipping)
✓ 2001 already present (skipping)


...

✓ 2006 already present (skipping)
✓ 2007 already present (skipping)
✓ 2008 already present (skipping)
✓ 2009 already present (skipping)
✓ 2010 already present (skipping)


✓ 2011 already present (skipping)
✓ 2012 already present (skipping)
✓ 2013 already present (skipping)
✓ 2014 already present (skipping)


✓ 2015 already present (skipping)
✓ 2016 already present (skipping)
✓ 2017 already present (skipping)
✓ 2018 already present (skipping)
✓ 2019 already present (skipping)


✓ 2020 already present (skipping)
✓ 2021 already present (skipping)
✓ 2022 already present (skipping)
✓ 2023 already present (skipping)


✓ 2024 already present (skipping)
✓ 2025 already present (skipping)
